# LLM Reasoning Prompt Design

This notebook documents how structured evidence packets are converted into LLM prompts. The final prompt builder lives in `src/prompts.py`; this notebook is the narrative/checking layer.

In [1]:
import json
from pathlib import Path

ROOT = Path('..')
evidence_path = ROOT / 'reports' / '03_evidence' / 'evidence_packets.json'

with open(evidence_path) as f:
    evidence_packets = json.load(f)

evidence_packets.keys()

dict_keys(['TP', 'FN', 'FP', 'TN'])

In [2]:
tp_packet = evidence_packets['TP']
print(json.dumps(tp_packet, indent=2)[:4000])

{
  "patient_label": "TP",
  "prediction": {
    "y_pred": 1,
    "y_proba": 0.998874831964749,
    "threshold": 0.7273530450209328,
    "y_true": 1,
    "prediction_type": "TP"
  },
  "risk_increasing_evidence": [
    {
      "feature": "d1_heartrate_min",
      "value": 0.0,
      "shap_value": 1.1382217805524988,
      "direction": "risk_increasing",
      "clinical_meaning": "extreme or very low heart rate may indicate severe instability or data quality issue",
      "caution_flags": [
        "Zero-valued vital sign; may reflect extreme clinical event or recording artifact."
      ]
    },
    {
      "feature": "d1_lactate_min",
      "value": 8.7,
      "shap_value": 0.7821194077129602,
      "direction": "risk_increasing",
      "clinical_meaning": "No predefined clinical interpretation available.",
      "caution_flags": []
    },
    {
      "feature": "d1_spo2_min",
      "value": 70.0,
      "shap_value": 0.5613483450223526,
      "direction": "risk_increasing",
      "clin

## Prompt construction rule

The LLM prompt receives only the model's own prediction and SHAP evidence. It does **not** receive the true label or TP/FN/FP/TN case type. Those fields remain in the evidence packet for internal audit/reporting only.

In [3]:
import sys
sys.path.append(str(ROOT))

from src.prompts import build_explanation_prompt

prompts = {
    case_type: build_explanation_prompt(packet)
    for case_type, packet in evidence_packets.items()
}

print(prompts['TP'][:3000])

You are a clinical AI explanation assistant.

Your task is to explain why the mortality prediction model made its prediction for this ICU patient.

Use only the provided evidence.
Do not invent clinical facts.
Do not mention or use the true label in the explanation.
Do not say the prediction was correct, incorrect, consistent with the true outcome, or inconsistent with the true outcome.
Do not add measurement units unless they are explicitly present in the evidence.
Do not invent normal ranges, diagnoses, mechanisms, or clinical details beyond the provided clinical_meaning.
If clinical_meaning is "No predefined clinical interpretation available.", do not infer a clinical interpretation for that feature.
For features without explicit clinical_meaning, only state that they increased or decreased the model's predicted risk.
For features without explicit clinical_meaning, do not describe the value as low, high, normal, adequate, stable, unstable, elevated, reduced, or moderate.
For feature

In [4]:
prompt_dir = ROOT / 'reports' / '04_llm_reasoning' / 'prompts'
prompt_dir.mkdir(parents=True, exist_ok=True)

for case_type, prompt in prompts.items():
    with open(prompt_dir / f'{case_type.lower()}_prompt.txt', 'w') as f:
        f.write(prompt)

sorted(path.name for path in prompt_dir.glob('*_prompt.txt'))

['fn_prompt.txt', 'fp_prompt.txt', 'tn_prompt.txt', 'tp_prompt.txt']

## Prompt safety constraints

The prompt explicitly instructs the LLM to:

- use only provided evidence,
- avoid invented clinical facts,
- avoid true-label leakage,
- avoid saying whether the prediction is correct/incorrect,
- avoid normal ranges or units unless present in evidence,
- preserve risk-increasing and risk-decreasing direction,
- mention caution flags when present,
- follow the five-section explanation structure.

This design makes the later deterministic validator meaningful because the expected output format and evidence boundaries are explicit.

In [5]:
for case_type, prompt in prompts.items():
    forbidden_prompt_terms = [term for term in ['True label', 'Case type', 'prediction_type'] if term in prompt]
    print(case_type, 'forbidden prompt terms:', forbidden_prompt_terms)

TP forbidden prompt terms: []
FN forbidden prompt terms: []
FP forbidden prompt terms: []
TN forbidden prompt terms: []


## Reasoning summary

The LLM reasoning layer is not given raw model internals directly. It receives a structured evidence packet translated into a strict prompt. This keeps the generated explanation anchored to local SHAP evidence and prevents the LLM from using hidden true-outcome information.